# CIFAR-10 CNN for Object Classification on the e-puck

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sakthi7sg/epuck-webots-cnn-object-detection/blob/main/cnn/cifar10_cnn.ipynb)

This notebook trains a convolutional neural network (CNN) on CIFAR-10 off the robot, evaluates it, tests it on my own photos and on an image from the e-puck camera in Webots, and measures how quantisation and pruning change its size, speed and accuracy.

| Section | Output for the brief |
|---|---|
| 3. Model | All layers of the model |
| 5. Evaluation | Test accuracy, per-class accuracy, confusion matrix |
| 6. Own images | Predictions on three of my own images |
| 7. Webots camera | Prediction on an image from the e-puck camera |
| 8. Optimisation | Size, latency and accuracy of FP32, int8 and pruned models |

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Training takes about 10 minutes on a T4.

## 1. Setup

In [ ]:
!pip -q install torchinfo

import copy, io, json, os, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchinfo import summary
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.family": "serif"})

print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

The model is defined in `cnn_model.py`. The Webots controller imports the same file, so the network trained here is exactly the network that runs on the robot.

In [ ]:
%%writefile cnn_model.py
"""CNN for CIFAR-10 (32 x 32 RGB images, 10 classes).

The same file is used by the training notebook and by the Webots controller,
so the network that is trained is exactly the network that runs on the robot.
"""
import torch
import torch.nn as nn

CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

# Per-channel mean and standard deviation of the CIFAR-10 training set.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


def conv_block(c_in, c_out, dropout):
    """Two 3x3 convolutions (each followed by batch norm and ReLU), then 2x2 max pooling."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),          # halves the height and width
        nn.Dropout(dropout),
    )


class CifarCNN(nn.Module):
    """VGG-style network: three convolution blocks and a two-layer classifier.

    Input 3 x 32 x 32 -> block 1 (32 ch, 16 x 16) -> block 2 (64 ch, 8 x 8)
    -> block 3 (128 ch, 4 x 4) -> fully connected 256 -> 10 class scores.
    """

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32, 0.2),
            conv_block(32, 64, 0.3),
            conv_block(64, 128, 0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def preprocess(rgb_uint8):
    """Turn an H x W x 3 uint8 RGB tensor into a normalised 1 x 3 x 32 x 32 batch.

    The largest centred square is cropped first, so the object is not squashed,
    and then resized to 32 x 32 with antialiasing.
    """
    img = rgb_uint8.permute(2, 0, 1).float().div(255.0)          # 3 x H x W, range 0..1
    _, h, w = img.shape
    s = min(h, w)
    top, left = (h - s) // 2, (w - s) // 2
    img = img[:, top:top + s, left:left + s]
    img = nn.functional.interpolate(img.unsqueeze(0), size=(32, 32), mode="bilinear",
                                    align_corners=False, antialias=True)
    mean = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(1, 3, 1, 1)
    return (img - mean) / std

In [ ]:
import importlib, cnn_model
importlib.reload(cnn_model)
from cnn_model import CifarCNN, CLASSES, CIFAR_MEAN, CIFAR_STD, preprocess

## 2. Data

CIFAR-10 has 60,000 colour images of 32 × 32 pixels in 10 classes: 50,000 for training and 10,000 for testing. The training images are augmented with random crops (4-pixel padding) and horizontal flips, which reduces overfitting. All images are normalised with the per-channel mean and standard deviation of the training set.

In [ ]:
BATCH = 128
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
test_tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

train_set = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=train_tf)
test_set = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=test_tf)
train_loader = torch.utils.data.DataLoader(train_set, BATCH, shuffle=True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_set, 256, shuffle=False, num_workers=2, pin_memory=True)
print(f"training images: {len(train_set):,} | test images: {len(test_set):,} | classes: {len(CLASSES)}")

def denorm(x):
    m, s = torch.tensor(CIFAR_MEAN).view(3, 1, 1), torch.tensor(CIFAR_STD).view(3, 1, 1)
    return (x.cpu() * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

raw = torchvision.datasets.CIFAR10("data", train=False, transform=test_tf)
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for c in range(10):
    idx = [i for i, y in enumerate(raw.targets) if y == c][:2]
    for r in range(2):
        axes[r, c].imshow(denorm(raw[idx[r]][0])); axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=9)
plt.suptitle("CIFAR-10 test images, two per class"); plt.tight_layout()
plt.savefig(f"{OUT}/cifar10_samples.png", bbox_inches="tight"); plt.show()

## 3. Model: all layers

A VGG-style network with three convolution blocks. Each block has two 3 × 3 convolutions with batch normalisation and ReLU, then 2 × 2 max pooling and dropout. The classifier flattens the 128 × 4 × 4 feature map and uses one hidden layer of 256 units.

In [ ]:
model = CifarCNN().to(DEVICE)
print(model)
print()
info = summary(model, input_size=(1, 3, 32, 32), col_names=("output_size", "num_params"),
               depth=3, device=DEVICE, verbose=0)
print(info)

In [ ]:
# Layer table for the report and slides
rows, x = [], torch.zeros(1, 3, 32, 32, device=DEVICE)
model.eval()
with torch.no_grad():
    for name, layer in model.named_modules():
        if len(list(layer.children())) == 0:
            x_in = x
            x = layer(x)
            n = sum(p.numel() for p in layer.parameters())
            rows.append((name, layer.__class__.__name__, tuple(x.shape[1:]), n))
print(f"{'#':>2}  {'layer':<14}{'type':<12}{'output shape':<18}{'parameters':>11}")
for i, (n, t, s, p) in enumerate(rows, 1):
    print(f"{i:>2}  {n:<14}{t:<12}{str(s):<18}{p:>11,}")
TOTAL_PARAMS = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {TOTAL_PARAMS:,}")

## 4. Training

| Setting | Value | Reason |
|---|---|---|
| Optimiser | SGD, momentum 0.9, Nesterov | Generalises well on CIFAR-10 |
| Learning rate | One-cycle, peak 0.1 | Fast, stable convergence |
| Weight decay | 5 × 10⁻⁴ | Regularisation |
| Loss | Cross-entropy with label smoothing 0.1 | Less over-confident predictions |
| Epochs | 30 | About 10 minutes on a T4 GPU |
| Batch size | 128 | |

In [ ]:
EPOCHS = 30
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1, epochs=EPOCHS,
                                                steps_per_epoch=len(train_loader))
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

def evaluate(net, loader, device=DEVICE):
    net.eval(); correct = total = 0; loss_sum = 0.0
    with torch.inference_mode():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = net(xb)
            loss_sum += F.cross_entropy(out, yb, reduction="sum").item()
            correct += (out.argmax(1) == yb).sum().item(); total += yb.numel()
    return loss_sum / total, correct / total

def train_epoch(net, opt, sched=None):
    net.train(); correct = total = 0; loss_sum = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            out = net(xb); loss = criterion(out, yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if sched: sched.step()
        loss_sum += loss.item() * yb.numel()
        correct += (out.argmax(1) == yb).sum().item(); total += yb.numel()
    return loss_sum / total, correct / total

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
best_acc, t0 = 0.0, time.time()
for epoch in range(1, EPOCHS + 1):
    tl, ta = train_epoch(model, optimizer, scheduler)
    vl, va = evaluate(model, test_loader)
    for k, v in zip(history, (tl, ta, vl, va)): history[k].append(v)
    if va > best_acc:
        best_acc = va; torch.save(model.state_dict(), "cifar10_cnn.pth")
    print(f"epoch {epoch:2d}/{EPOCHS} | train loss {tl:.3f} acc {ta*100:5.2f}% | "
          f"test loss {vl:.3f} acc {va*100:5.2f}% | {time.time()-t0:5.0f} s")
TRAIN_TIME = time.time() - t0
model.load_state_dict(torch.load("cifar10_cnn.pth", map_location=DEVICE))
print(f"\nBest test accuracy: {best_acc*100:.2f}%  (training time {TRAIN_TIME/60:.1f} min)")

In [ ]:
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ep, history["train_loss"], label="training"); ax[0].plot(ep, history["test_loss"], label="test")
ax[0].set(xlabel="epoch", ylabel="loss", title="Loss"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, np.array(history["train_acc"]) * 100, label="training")
ax[1].plot(ep, np.array(history["test_acc"]) * 100, label="test")
ax[1].set(xlabel="epoch", ylabel="accuracy (%)", title="Accuracy"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT}/training_curves.png", bbox_inches="tight"); plt.show()

## 5. Evaluation on the 10,000 test images

In [ ]:
model.eval(); preds, labels = [], []
with torch.inference_mode():
    for xb, yb in test_loader:
        preds.append(model(xb.to(DEVICE)).argmax(1).cpu()); labels.append(yb)
preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
TEST_ACC = (preds == labels).mean()
print(f"Test accuracy: {TEST_ACC*100:.2f}%  ({(preds == labels).sum():,} of {len(labels):,} correct)\n")
print(classification_report(labels, preds, target_names=CLASSES, digits=3))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
ax.set_title(f"Confusion matrix (test accuracy {TEST_ACC*100:.1f}%)")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_matrix.png", bbox_inches="tight"); plt.show()

per_class = cm.diagonal() / cm.sum(1)
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"Hardest class: {CLASSES[per_class.argmin()]} ({per_class.min()*100:.1f}%) | "
      f"easiest: {CLASSES[per_class.argmax()]} ({per_class.max()*100:.1f}%)")
print(f"Most common confusion: {CLASSES[i]} predicted as {CLASSES[j]} ({off[i, j]} images)")

In [ ]:
# Some test images the model got wrong
wrong = np.where(preds != labels)[0][:16]
fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for a, k in zip(axes.flat, wrong):
    a.imshow(denorm(test_set[k][0])); a.axis("off")
    a.set_title(f"true: {CLASSES[labels[k]]}\npred: {CLASSES[preds[k]]}", fontsize=8, color="#b22222")
plt.suptitle("Misclassified test images"); plt.tight_layout()
plt.savefig(f"{OUT}/misclassified.png", bbox_inches="tight"); plt.show()

**Comparison with published results on CIFAR-10** (test accuracy):

| Model | Accuracy | Parameters |
|---|---|---|
| Random guess | 10% | – |
| LeNet-5 style CNN (2 conv layers) | about 65–70% | about 62 k |
| **This model (6 conv layers)** | *see the next cell* | about 0.8 M |
| ResNet-18 (He et al., 2016) | about 93–95% | 11 M |

In [ ]:
print(f"This model: {TEST_ACC*100:.2f}% with {TOTAL_PARAMS/1e6:.2f} M parameters "
      f"({TOTAL_PARAMS/11.2e6*100:.0f}% of the parameters of ResNet-18)")

## 6. Predictions on three of my own images

Run the cell and upload three photos taken with my phone (any of the 10 classes). Each image is centre-cropped to a square and resized to 32 × 32, the same as the Webots controller does.

In [ ]:
from PIL import Image

def classify_rgb(rgb, net=model, device=DEVICE):
    """rgb: H x W x 3 uint8 array. Returns class probabilities (10,)."""
    x = preprocess(torch.from_numpy(np.ascontiguousarray(rgb))).to(device)
    net.eval()
    with torch.inference_mode():
        return F.softmax(net(x), 1)[0].cpu().numpy()

def show_predictions(paths, title, out_name):
    fig, axes = plt.subplots(len(paths), 3, figsize=(10, 3.1 * len(paths)),
                             gridspec_kw={"width_ratios": [1.4, 1, 1.6]}, squeeze=False)
    results = []
    for row, p in zip(axes, paths):
        rgb = np.array(Image.open(p).convert("RGB"))
        prob = classify_rgb(rgb)
        top = prob.argsort()[::-1][:3]
        small = denorm(preprocess(torch.from_numpy(rgb))[0])
        row[0].imshow(rgb); row[0].set_title(os.path.basename(p), fontsize=9); row[0].axis("off")
        row[1].imshow(small, interpolation="nearest"); row[1].set_title("model input, 32 × 32", fontsize=9); row[1].axis("off")
        row[2].barh([CLASSES[t] for t in top][::-1], prob[top][::-1] * 100, color=["#8FB8DE", "#8FB8DE", "#1F4E79"])
        row[2].set_xlim(0, 100); row[2].set_xlabel("probability (%)")
        row[2].set_title(f"prediction: {CLASSES[top[0]]} ({prob[top[0]]*100:.1f}%)", fontsize=10, fontweight="bold")
        results.append({"image": os.path.basename(p), "prediction": CLASSES[top[0]],
                        "confidence": round(float(prob[top[0]]), 4),
                        "top3": [(CLASSES[t], round(float(prob[t]), 4)) for t in top]})
    plt.suptitle(title); plt.tight_layout()
    plt.savefig(f"{OUT}/{out_name}", bbox_inches="tight"); plt.show()
    return results

def upload_images(folder):
    os.makedirs(folder, exist_ok=True)
    try:
        from google.colab import files
        up = files.upload()
        paths = []
        for name, data in up.items():
            p = os.path.join(folder, name); open(p, "wb").write(data); paths.append(p)
        return paths
    except ImportError:                        # not in Colab: use the images already in the folder
        return sorted(os.path.join(folder, f) for f in os.listdir(folder)
                      if f.lower().endswith((".png", ".jpg", ".jpeg")))

own_paths = upload_images("my_images")
OWN_RESULTS = show_predictions(own_paths, "Predictions on my own images", "own_image_predictions.png")
for r in OWN_RESULTS: print(r["image"], "->", r["prediction"], f"{r['confidence']*100:.1f}%")

## 7. Prediction on an image from the e-puck camera in Webots

Run the controller `controllers/cnn_camera_classifier` in Webots. It saves the camera frames it classifies as `camera_frame_*.png` in the controller folder. Upload one or more of them here. The camera gives 52 × 39 pixels, so the image is cropped to 39 × 39 and resized to 32 × 32.

In [ ]:
cam_paths = upload_images("webots_frames")
WEBOTS_RESULTS = show_predictions(cam_paths, "Predictions on e-puck camera images from Webots",
                                  "webots_camera_predictions.png")
for r in WEBOTS_RESULTS: print(r["image"], "->", r["prediction"], f"{r['confidence']*100:.1f}%")

## 8. Optimisation for running on the robot

The e-puck controller runs every 64 ms. If one classification takes longer than that, the motors are updated late and the robot's movement suffers. Four options are measured on the **CPU with one thread**, which is closer to a small robot computer than a GPU:

1. **FP32 baseline**, the trained model as it is.
2. **Dynamic int8 quantisation**: fully connected weights stored as 8-bit integers.
3. **Static int8 quantisation**: weights *and* activations in 8-bit, calibrated on 2,000 training images.
4. **Pruning**: the 50% smallest weights set to zero, then fine-tuned for 3 epochs.

In [ ]:
from torch.ao.quantization import quantize_dynamic, get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torch.nn.utils.prune as prune

engines = torch.backends.quantized.supported_engines
torch.backends.quantized.engine = "x86" if "x86" in engines else ("fbgemm" if "fbgemm" in engines else "qnnpack")
cpu = torch.device("cpu")
fp32 = copy.deepcopy(model).to(cpu).eval()

def size_mb(net):
    buf = io.BytesIO(); torch.save(net.state_dict(), buf); return buf.tell() / 1e6

def latency_ms(net, runs=300):
    torch.set_num_threads(1); x = torch.randn(1, 3, 32, 32); net.eval()
    with torch.inference_mode():
        for _ in range(20): net(x)
        t = time.perf_counter()
        for _ in range(runs): net(x)
    torch.set_num_threads(os.cpu_count())
    return (time.perf_counter() - t) / runs * 1000

def nonzero_params(net):
    return sum(int((p != 0).sum()) for n, p in net.named_parameters() if "weight" in n)

# 2. dynamic int8
dyn = quantize_dynamic(copy.deepcopy(fp32), {nn.Linear}, dtype=torch.qint8)

# 3. static int8 (post-training quantisation, FX graph mode)
calib_loader = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10("data", train=True, transform=test_tf), batch_size=100, shuffle=True)
prepared = prepare_fx(copy.deepcopy(fp32), get_default_qconfig_mapping(torch.backends.quantized.engine),
                      example_inputs=(torch.randn(1, 3, 32, 32),))
with torch.inference_mode():
    for k, (xb, _) in enumerate(calib_loader):
        prepared(xb)
        if k == 19: break                      # 20 batches = 2,000 images
static = convert_fx(prepared)

# 4. pruning: global L1 unstructured, 50%, then fine-tune 3 epochs with the mask kept
pruned = copy.deepcopy(model).to(DEVICE)
targets = [(m, "weight") for m in pruned.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
prune.global_unstructured(targets, pruning_method=prune.L1Unstructured, amount=0.5)
_, acc_pruned_before = evaluate(pruned, test_loader)
ft_opt = torch.optim.SGD(pruned.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
for e in range(3):
    train_epoch(pruned, ft_opt)
for m, n in targets: prune.remove(m, n)
pruned = pruned.to(cpu).eval()
print(f"Pruned accuracy before fine-tuning: {acc_pruned_before*100:.2f}%")

In [ ]:
variants = [("FP32 baseline", fp32), ("Dynamic int8", dyn), ("Static int8", static), ("Pruned 50% + fine-tune", pruned)]
OPT_RESULTS = []
for name, net in variants:
    _, acc = evaluate(net, test_loader, device=cpu)
    ms = latency_ms(net)
    OPT_RESULTS.append({"model": name, "accuracy_%": round(acc * 100, 2), "size_MB": round(size_mb(net), 2),
                        "latency_ms": round(ms, 2), "fps": round(1000 / ms, 1),
                        "share_of_64ms_step_%": round(ms / 64 * 100, 1)})
base = OPT_RESULTS[0]
print(f"{'model':<24}{'accuracy':>9}{'size MB':>9}{'ms/image':>10}{'FPS':>8}{'of 64 ms':>10}{'speed-up':>10}")
for r in OPT_RESULTS:
    print(f"{r['model']:<24}{r['accuracy_%']:>8.2f}%{r['size_MB']:>9.2f}{r['latency_ms']:>10.2f}"
          f"{r['fps']:>8.0f}{r['share_of_64ms_step_%']:>9.1f}%{base['latency_ms']/r['latency_ms']:>9.2f}x")
print(f"\nNon-zero weights: FP32 {nonzero_params(fp32):,} | pruned {nonzero_params(pruned):,}")
print("Note: unstructured pruning keeps the same tensor shapes, so file size and dense CPU speed "
      "only improve with sparse storage or sparse kernels.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
names = [r["model"].replace(" + ", "\n+ ") for r in OPT_RESULTS]
cols = ["#1F4E79", "#8FB8DE", "#5FA857", "#F6C453"]
ax[0].bar(names, [r["latency_ms"] for r in OPT_RESULTS], color=cols)
ax[0].set(ylabel="ms per image (1 CPU thread)", title="Inference time"); ax[0].tick_params(axis="x", labelsize=8)
ax[1].bar(names, [r["size_MB"] for r in OPT_RESULTS], color=cols)
ax[1].set(ylabel="MB", title="Model size"); ax[1].tick_params(axis="x", labelsize=8)
for a, key in zip(ax, ("latency_ms", "size_MB")):
    a.set_ylim(0, max(r[key] for r in OPT_RESULTS) * 1.3)
    for k, r in enumerate(OPT_RESULTS):
        a.text(k, r[key], f"{r[key]:.2f}\n{r['accuracy_%']:.1f}%", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/optimisation_comparison.png", bbox_inches="tight"); plt.show()

**Running the CNN less often.** The colour search does not need a class label every 64 ms. Classifying one frame every *N* control steps divides the CPU load by *N*:

In [ ]:
fast = min(OPT_RESULTS, key=lambda r: r["latency_ms"])
print(f"{'classify every':<16}{'interval':>10}{'FP32 load':>12}{fast['model'] + ' load':>24}")
for n in (1, 4, 8, 16):
    print(f"{n:>3} steps{'':<7}{n*64:>8} ms{base['latency_ms']/(n*64)*100:>11.1f}%{fast['latency_ms']/(n*64)*100:>23.1f}%")

## 9. Export for Webots

In [ ]:
torch.save(fp32.state_dict(), "cifar10_cnn.pth")
results = {"test_accuracy_%": round(float(TEST_ACC) * 100, 2), "best_epoch_accuracy_%": round(best_acc * 100, 2),
           "parameters": TOTAL_PARAMS, "epochs": EPOCHS, "training_minutes": round(TRAIN_TIME / 60, 1),
           "per_class_accuracy_%": {c: round(float(a) * 100, 1) for c, a in zip(CLASSES, per_class)},
           "own_images": OWN_RESULTS, "webots_camera": WEBOTS_RESULTS, "optimisation": OPT_RESULTS}
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)
!cp cifar10_cnn.pth cnn_model.py {OUT}/ && zip -qr cnn_outputs.zip {OUT}
print("Saved cifar10_cnn.pth and cnn_outputs.zip (figures, results.json, model)")
try:
    from google.colab import files
    files.download("cifar10_cnn.pth"); files.download("cnn_outputs.zip")
except ImportError:
    pass

Copy `cifar10_cnn.pth` into `controllers/cnn_camera_classifier/` in the Webots project. The figures in `cnn_outputs.zip` are used in the presentation and the report.

## 10. Summary

In [ ]:
print(f"Test accuracy           {TEST_ACC*100:.2f}%   ({TOTAL_PARAMS:,} parameters, {EPOCHS} epochs)")
print(f"Own images              " + ", ".join(f"{r['image']}: {r['prediction']}" for r in OWN_RESULTS))
print(f"Webots camera           " + ", ".join(f"{r['image']}: {r['prediction']}" for r in WEBOTS_RESULTS))
for r in OPT_RESULTS:
    print(f"{r['model']:<24}{r['accuracy_%']:.2f}%  {r['latency_ms']:.2f} ms  {r['size_MB']:.2f} MB")